# Step 1: Research & Data Source Discovery

## 1.1 Environment Setup and Dependencies

We fetch the data from OpenStreetMap. We use the original OSM ID (osmid) as our primary identifier and calculate the exact center point (latitude and longitude) for each location.

* **Primary Source: OpenStreetMap (OSM)**: Used to extract the spatial location of employment agencies.


## 1.2 Data and Boundary Configuration

The project focuses exclusively on data within the **Berlin, Germany** boundary.

* **Spatial Integrity Plan**: Data will be joined to the **Local Reference System (LOR) boundaries** to derive the mandatory `district_id` and `neighborhood_id` for final database compliance.

In [197]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import os 
import json

# --- 1.1 CONFIGURATION ---
# Using the specific paths and tags from your previous workflow
PLACE_NAME = "Berlin, Germany"
OSM_TAGS = {"office": "employment_agency"}

# Update LOR_PATH to match the exact filename of the GeoJSON you uploaded
#LOR_PATH = "lor_ortsteile (1).geojson" 
#OUTPUT_PATH = "output/jobcenters_berlin.csv"

print("Libraries loaded.")
print(f"Configuration set for {PLACE_NAME} with OSM tags: {OSM_TAGS}")

# --- 1.2 LIVE DATA EXTRACTION (OSM) ---
print("\nFetching live data from OpenStreetMap (Overpass API)...")
try:
    # Fetch data and ensure the coordinate system is standard WGS84 (EPSG:4326)
    jobcenter_data_raw = ox.features_from_place(PLACE_NAME, OSM_TAGS)
    jobcenter_data_raw = gpd.GeoDataFrame(
        jobcenter_data_raw,
        geometry="geometry",
        crs="EPSG:4326"
    )
    print(f"Success! Retrieved {len(jobcenter_data_raw)} features.")
except Exception as e:
    raise RuntimeError(f"OSM extraction failed: {e}")

# --- 1.3 MANDATORY DATA CLEANING ---
# We explicitly check and report on null values in mandatory columns
print("\n--- Diagnostic Check: Nulls in Critical Columns ---")
null_counts = jobcenter_data_raw[['name', 'geometry']].isnull().sum()
print("Missing values in critical columns:")
print(null_counts)

# Drop rows missing 'name' or 'geometry' to enforce database NOT NULL compliance
initial_count = len(jobcenter_data_raw)
jobcenter_enriched = jobcenter_data_raw.dropna(subset=["name", "geometry"]).copy()

dropped_count = initial_count - len(jobcenter_enriched)
print(f"Mandatory Drop: Removed {dropped_count} rows due to missing name/geometry.")

# --- 1.4 COORDINATE PREPARATION ---
# Extract centroids to handle both 'Point' and 'Polygon' features safely
jobcenter_enriched['latitude'] = jobcenter_enriched.geometry.centroid.y
jobcenter_enriched['longitude'] = jobcenter_enriched.geometry.centroid.x

print("\n--- Step 1 Complete ---")
print(jobcenter_enriched[['name', 'latitude', 'longitude']].head())

Libraries loaded.
Configuration set for Berlin, Germany with OSM tags: {'office': 'employment_agency'}

Fetching live data from OpenStreetMap (Overpass API)...
Success! Retrieved 65 features.

--- Diagnostic Check: Nulls in Critical Columns ---
Missing values in critical columns:
name        2
geometry    0
dtype: int64
Mandatory Drop: Removed 2 rows due to missing name/geometry.

--- Step 1 Complete ---
                                               name   latitude  longitude
element id                                                               
node    275368512   Jobcenter Mitte am Leopoldplatz  52.546772  13.356516
        1211913324           Arbeitsagentur Spandau  52.533775  13.186554
        1340158173        Jobcenter Berlin Neukölln  52.478975  13.427887
        1450906609               Agentur für Arbeit  52.578452  13.308718
        2277566662               Agentur für Arbeit  52.456592  13.411478


In [198]:
jobcenter_data_raw.head()

geometry addr:city addr:country  \
element id                                                             
node    275368512   POINT (13.35652 52.54677)    Berlin           DE   
        1211913324  POINT (13.18655 52.53378)    Berlin           DE   
        1340158173  POINT (13.42789 52.47898)    Berlin           DE   
        1450906609  POINT (13.30872 52.57845)    Berlin           DE   
        2277566662  POINT (13.41148 52.45659)       NaN          NaN   

                                     addr:housename addr:housenumber  \
element id                                                             
node    275368512   Jobcenter Mitte am Leopoldplatz              147   
        1211913324                              NaN            75-77   
        1340158173                              NaN               27   
        1450906609                              NaN               40   
        2277566662                              NaN              NaN   

                   addr:postcode         addr:street  addr:suburb  \
element id                                                          
node    275368512          13353        Müllerstraße      Wedding   
        1211913324         13581  Brunsbütteler Damm      Spandau   
        1340158173         12053      Mainzer Straße     Neukölln   
        1450906609         13509       Innungsstraße  Borsigwalde   
        2277566662           NaN                 NaN          NaN   

                                 brand brand:wikidata  ... lda:criteria  \
element id                                             ...                
node    275368512            Jobcenter      Q56292847  ...          NaN   
        1211913324                 NaN            NaN  ...          NaN   
        1340158173           Jobcenter      Q56292847  ...          NaN   
        1450906609  Agentur für Arbeit       Q1478016  ...          NaN   
        2277566662  Agentur für Arbeit       Q1478016  ...          NaN   

                   ref:lda height air_conditioning toilets alt_name name:de  \
element id                                                                    
node    275368512      NaN    NaN              NaN     NaN      NaN     NaN   
        1211913324     NaN    NaN              NaN     NaN      NaN     NaN   
        1340158173     NaN    NaN              NaN     NaN      NaN     NaN   
        1450906609     NaN    NaN              NaN     NaN      NaN     NaN   
        2277566662     NaN    NaN              NaN     NaN      NaN     NaN   

                   type government wikidata  
element id                                   
node    275368512   NaN        NaN      NaN  
        1211913324  NaN        NaN      NaN  
        1340158173  NaN        NaN      NaN  
        1450906609  NaN        NaN      NaN  
        2277566662  NaN        NaN      NaN  

[5 rows x 62 columns]

In [199]:
jobcenter_data_raw = jobcenter_data_raw.reset_index()

In [200]:
jobcenter_data_raw.columns

Index(['element', 'id', 'geometry', 'addr:city', 'addr:country',
       'addr:housename', 'addr:housenumber', 'addr:postcode', 'addr:street',
       'addr:suburb', 'brand', 'brand:wikidata', 'check_date:opening_hours',
       'contact:phone', 'contact:website', 'name', 'office', 'opening_hours',
       'wheelchair', 'toilets:wheelchair', 'website', 'branch', 'check_date',
       'email', 'opening_hours:signed', 'operator', 'phone', 'brand:wikipedia',
       'internet_access', 'internet_access:fee', 'internet_access:ssid',
       'official_name', 'smoking', 'source', 'contact:email', 'contact:fax',
       'description', 'operator:type', 'short_name', 'level', 'note',
       'addr:floor', 'building:levels', 'image', 'building', 'building:colour',
       'roof:levels', 'roof:shape', 'old_name', 'landuse', 'addr:place',
       'heritage', 'heritage:operator', 'heritage:website', 'lda:criteria',
       'ref:lda', 'height', 'air_conditioning', 'toilets', 'alt_name',
       'name:de', 'type',

In [201]:
jobcenter_enriched.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 63 entries, ('node', np.int64(275368512)) to ('way', np.int64(1092876804))
Data columns (total 64 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   geometry                  63 non-null     geometry
 1   addr:city                 50 non-null     object  
 2   addr:country              28 non-null     object  
 3   addr:housename            2 non-null      object  
 4   addr:housenumber          51 non-null     object  
 5   addr:postcode             50 non-null     object  
 6   addr:street               50 non-null     object  
 7   addr:suburb               27 non-null     object  
 8   brand                     27 non-null     object  
 9   brand:wikidata            27 non-null     object  
 10  check_date:opening_hours  12 non-null     object  
 11  contact:phone             4 non-null      object  
 12  contact:website           6 non-null      obj

## Step 2: Cleanup & Removing Redundancy
Explanation: Here we drop city and country because they are redundant for a Berlin project. We also remove contact:website and operator:type to keep the schema lean.
Why drop contact "website"

Maintenance: External URLs like websites change frequently. If you include them in the primary table now, the data becomes "stale" very quickly.

Scope: The current goal is to map the job centers to the Berlin District LOR system. Extra information like websites or phone numbers can be added in a later "enrichment" task once the primary table structure is approved.

Additionally: The operator:type column is a classification tag in OpenStreetMap. It tells the database who runs the facility. In the context of Berlin Job Centers, this usually indicates public. 
The center is a government-run entity (e.g., the Bundesagentur für Arbeit or local municipal government). Most Job Centers fall into this category.

In [202]:
pip install geopy

Note: you may need to restart the kernel to use updated packages.


In [203]:
print(jobcenter_clean.columns.tolist())

['element', 'id', 'geometry', 'addr:city', 'addr:country', 'addr:housename', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'brand', 'brand:wikidata', 'check_date:opening_hours', 'contact:phone', 'contact:website', 'center_name', 'office', 'opening_hours', 'wheelchair', 'toilets:wheelchair', 'website', 'branch', 'check_date', 'email', 'opening_hours:signed', 'operator', 'phone', 'brand:wikipedia', 'internet_access', 'internet_access:fee', 'internet_access:ssid', 'official_name', 'smoking', 'source', 'contact:email', 'contact:fax', 'description', 'operator:type', 'short_name', 'level', 'note', 'addr:floor', 'building:levels', 'image', 'building', 'building:colour', 'roof:levels', 'roof:shape', 'old_name', 'landuse', 'addr:place', 'heritage', 'heritage:operator', 'heritage:website', 'lda:criteria', 'ref:lda', 'height', 'air_conditioning', 'toilets', 'alt_name', 'name:de', 'type', 'government', 'wikidata', 'latitude', 'longitude', 'address', 'postal_code', 'district_le

In [204]:
from geopy.geocoders import Nominatim
from time import sleep

# 1. INITIAL CLEANUP: Rename and prepare coordinates
# We use jobcenter_mapped (the result of your spatial join)
jobcenter_clean = jobcenter_mapped.copy()
jobcenter_clean = jobcenter_clean.rename(columns={'name': 'center_name'})

# Calculate centroids to ensure we have lat/lon for both Points and Polygons
centroids = jobcenter_clean.geometry.centroid
jobcenter_clean['latitude'] = centroids.y
jobcenter_clean['longitude'] = centroids.x

# 2. BUILD ADDRESS FROM COLUMNS (The Boss's First Priority)
# We use fillna('') to avoid "NaN" appearing in the text strings
jobcenter_clean['address'] = (
    jobcenter_clean['addr:street'].fillna('') + ' ' + 
    jobcenter_clean['addr:housenumber'].fillna('')
).str.strip()

# Add house name in brackets if it exists (e.g., "Jobcenter Mitte")
mask_housename = jobcenter_clean['addr:housename'].notna()
jobcenter_clean.loc[mask_housename, 'address'] = (
    jobcenter_clean['address'] + ' (' + jobcenter_clean['addr:housename'] + ')'
).str.strip()

# Map the postal code from OSM
jobcenter_clean['postal_code'] = jobcenter_clean['addr:postcode']

# 3. NOMINATIM FALLBACK (The Boss's Second Priority)
geolocator = Nominatim(user_agent="berlin_jobcenter_locator")

def get_nominatim_data(lat, lon):
    """Retrieves both address and postcode from Nominatim"""
    try:
        location = geolocator.reverse((lat, lon), exactly_one=True, language='de')
        sleep(1) # Crucial: Respect Nominatim's 1-second rate limit
        if location:
            address_text = location.address
            postcode = location.raw.get('address', {}).get('postcode')
            return address_text, postcode
        return None, None
    except:
        return None, None

# Find rows where address is still empty OR postal_code is NaN
mask_missing = (jobcenter_clean['address'] == "") | (jobcenter_clean['postal_code'].isna())

if mask_missing.any():
    print(f"🔍 Found {mask_missing.sum()} rows needing Nominatim enrichment. Starting fallback...")
    
    # We apply the function to fill both columns at once
    results = jobcenter_clean[mask_missing].apply(
        lambda row: get_nominatim_data(row['latitude'], row['longitude']), axis=1
    )
    
    # Extract the results back into the dataframe
    jobcenter_clean.loc[mask_missing, 'address'] = [r[0] for r in results]
    jobcenter_clean.loc[mask_missing, 'postal_code'] = [r[1] for r in results]
else:
    print(" All addresses and postal codes were successfully built from existing data!")

print("\n--- Verification of Enriched Data ---")
print(jobcenter_clean[['center_name', 'address', 'postal_code']].head())

🔍 Found 15 rows needing Nominatim enrichment. Starting fallback...

--- Verification of Enriched Data ---
                       center_name  \
0  Jobcenter Mitte am Leopoldplatz   
1           Arbeitsagentur Spandau   
2        Jobcenter Berlin Neukölln   
3               Agentur für Arbeit   
4               Agentur für Arbeit   

                                             address postal_code  
0  Müllerstraße 147 (Jobcenter Mitte am Leopoldpl...       13353  
1                           Brunsbütteler Damm 75-77       13581  
2                                  Mainzer Straße 27       12053  
3                                   Innungsstraße 40       13509  
4  Agentur für Arbeit, 43-44, Gottlieb-Dunkel-Str...       12099  


## Step 3: Spatial Mapping (District Join)

Explanation: Load the official Berlin district file and  use a Spatial Join to see which district polygon each job center point "falls into." This gives us the neighborhood and district names automatically.

In [205]:
LOR_PATH = "lor_ortsteile.geojson"
lor_gdf = gpd.read_file(LOR_PATH).to_crs(epsg=4326)

In [206]:
import os
print("LOR file exists:", os.path.exists(LOR_PATH))

LOR file exists: True


In [208]:
import geopandas as gpd

# 1. Rename columns based on the 'lor_ortsteile' properties found in the file
lor_gdf = lor_gdf.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
})

# 2. Spatial Join: Mapping Job Center points to District polygons
# This uses the cleaned 'jobcenter_clean' data from your previous cell
jobcenter_mapped = gpd.sjoin(
    jobcenter_clean.reset_index(drop=True), 
    lor_gdf[['district', 'neighborhood', 'neighborhood_id', 'geometry']], 
    how='left', 
    predicate='within'
)

# --- Verification ---
print("--- Check Mapping Results ---")
print(jobcenter_mapped['district'].value_counts())

print("\n--- Check Mapped Data Preview ---")
print(jobcenter_mapped[['center_name', 'district', 'neighborhood']].head())

ValueError: 'index_right' cannot be a column name in the frames being joined

## 4: Stable ID Generation and District Mapping
Deterministic Stable ID: A persistent, numeric-only ID is generated using hashlib.sha256. By hashing the geographic centroid, we ensure IDs are unique and immutable, avoiding previous AttributeError issues with different geometry types.

Official District Mapping: To comply with the final data pool schema, we map administrative district names to their official 8-digit numeric IDs (e.g., Mitte = 11001001). This ensures the data is ready for SQL relational joins.hment:** The `enrich_data_from_wikidata` function is applied to fill the `operator_name` and `contact_website` columns.

In [ ]:
jobcenter_mapped.head()

,element,id,geometry,addr:city,addr:country,addr:housename,addr:housenumber,addr:postcode,addr:street,addr:suburb,...,district_left,neighborhood_left,neighborhood_id_left,district_right,neighborhood_right,neighborhood_id_right,index_right,district,neighborhood,neighborhood_id
0,node,275368512,POINT (13.35652 52.54677),Berlin,DE,Jobcenter Mitte am Leopoldplatz,147,13353,Müllerstraße,Wedding,...,Mitte,Wedding,0105,Mitte,Wedding,0105,4,Mitte,Wedding,0105
1,node,1211913324,POINT (13.18655 52.53378),Berlin,DE,NaN,75-77,13581,Brunsbütteler Damm,Spandau,...,Spandau,Spandau,0501,Spandau,Spandau,0501,28,Spandau,Spandau,0501
2,node,1340158173,POINT (13.42789 52.47898),Berlin,DE,NaN,27,12053,Mainzer Straße,Neukölln,...,Neukölln,Neukölln,0801,Neukölln,Neukölln,0801,50,Neukölln,Neukölln,0801
3,node,1450906609,POINT (13.30872 52.57845),Berlin,DE,NaN,40,13509,Innungsstraße,Borsigwalde,...,Reinickendorf,Borsigwalde,1211,Reinickendorf,Borsigwalde,1211,95,Reinickendorf,Borsigwalde,1211
4,node,2277566662,POINT (13.41148 52.45659),NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Tempelhof-Schöneberg,Tempelhof,0703,Tempelhof-Schöneberg,Tempelhof,0703,46,Tempelhof-Schöneberg,Tempelhof,0703


In [ ]:
import hashlib

# --- 4.1 DEFINITIONS (Write once) ---
def generate_stable_id(name, lat, lon):
    """Generates a unique 10-digit ID based on name and coordinates."""
    input_data = f"{name}_{lat}_{lon}".encode('utf-8')
    hash_hex = hashlib.sha256(input_data).hexdigest()
    return int(hash_hex, 16) % (10**10)

district_mapping = {
    'Mitte': '11001001', 'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003', 'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005', 'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007', 'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009', 'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011', 'Reinickendorf': '11012012'
}

# --- 4.2 EXECUTION (The Calls) ---
# 1. Coordinate Prep (Ensuring columns exist)
jobcenter_mapped['latitude'] = jobcenter_mapped.geometry.centroid.y
jobcenter_mapped['longitude'] = jobcenter_mapped.geometry.centroid.x

# 2. Call the Stable ID function
print("Generating stable IDs...")
jobcenter_mapped['id'] = jobcenter_mapped.apply(
    lambda row: generate_stable_id(row['center_name'], row['latitude'], row['longitude']), 
    axis=1
)

# 3. Call the District mapping
print("Mapping districts...")
jobcenter_mapped['district_id'] = jobcenter_mapped['district'].map(district_mapping)

print("Step 4 complete. Data is enriched and identified.")
print("Mapping district names to official IDs...")
jobcenter_mapped['district_id'] = jobcenter_mapped['district'].map(district_mapping).astype(str)

# --- Verification ---
print("\n--- Step 4 Verification ---")
print(jobcenter_mapped[['id', 'center_name', 'district', 'district_id']].head())

Generating stable IDs...
Mapping districts...
Step 4 complete. Data is enriched and identified.
Mapping district names to official IDs...

--- Step 4 Verification ---
           id                      center_name              district  \
0  6660665090  Jobcenter Mitte am Leopoldplatz                 Mitte   
1  1092468394           Arbeitsagentur Spandau               Spandau   
2   730832232        Jobcenter Berlin Neukölln              Neukölln   
3   246338546               Agentur für Arbeit         Reinickendorf   
4  6239357044               Agentur für Arbeit  Tempelhof-Schöneberg   

  district_id  
0    11001001  
1    11005005  
2    11008008  
3    11012012  
4    11007007  


In [ ]:
jobcenter_mapped.head()

,element,id,geometry,addr:city,addr:country,addr:housename,addr:housenumber,addr:postcode,addr:street,addr:suburb,...,neighborhood_left,neighborhood_id_left,district_right,neighborhood_right,neighborhood_id_right,index_right,district,neighborhood,neighborhood_id,district_id
0,node,6660665090,POINT (13.35652 52.54677),Berlin,DE,Jobcenter Mitte am Leopoldplatz,147,13353,Müllerstraße,Wedding,...,Wedding,0105,Mitte,Wedding,0105,4,Mitte,Wedding,0105,11001001
1,node,1092468394,POINT (13.18655 52.53378),Berlin,DE,NaN,75-77,13581,Brunsbütteler Damm,Spandau,...,Spandau,0501,Spandau,Spandau,0501,28,Spandau,Spandau,0501,11005005
2,node,730832232,POINT (13.42789 52.47898),Berlin,DE,NaN,27,12053,Mainzer Straße,Neukölln,...,Neukölln,0801,Neukölln,Neukölln,0801,50,Neukölln,Neukölln,0801,11008008
3,node,246338546,POINT (13.30872 52.57845),Berlin,DE,NaN,40,13509,Innungsstraße,Borsigwalde,...,Borsigwalde,1211,Reinickendorf,Borsigwalde,1211,95,Reinickendorf,Borsigwalde,1211,11012012
4,node,6239357044,POINT (13.41148 52.45659),NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Tempelhof,0703,Tempelhof-Schöneberg,Tempelhof,0703,46,Tempelhof-Schöneberg,Tempelhof,0703,11007007


## 5: Data Standardization and Final Export
Schema Compliance: The final dataset is filtered to include only the mandatory 8 columns required for the database pool: id, district_id, center_name, latitude, longitude, neighborhood, district, and neighborhood_id.

WKT & Coordinate Prep: Coordinates are extracted from the geometric centroids and formatted as numeric floats, ensuring compatibility with standard SQL spatial types.

Stable ID Integration: The deterministic IDs generated in Step 4 are finalized as the primary keys for this dataset.

Data Source Attribution: A data_source tag (OSM_LOR) is appended to ensure traceability for future audits.

In [ ]:
import os

# --- 5.1 SPATIAL DATA PREP ---
# 1. Safety check: Remove any rows with missing shapes
df_ready = jobcenter_mapped.dropna(subset=['geometry']).copy()

# 2. Keep geometry as text (WKT) so the SQL 'to_sql' command can handle it
# We keep the name 'geometry' as you requested
df_ready['geometry'] = df_ready['geometry'].apply(
    lambda x: x.wkt if x is not None else None
)

# --- 5.2 FINAL SCHEMA SELECTION ---
# These names match your boss's requirements for the production layer
final_columns = [
    'id', 
    'district_id', 
    'center_name', 
    'address', 
    'postal_code', 
    'latitude', 
    'longitude', 
    'geometry',      
    'neighborhood', 
    'district', 
    'neighborhood_id'
]

# Create the final dataframe and add the source tag
df_final = df_ready[final_columns].copy()
df_final['data_source'] = 'OSM_LOR'

# --- 5.3 VERIFICATION & EXPORT ---
print("--- Final Production Audit ---")
print(f"Total Records: {len(df_final)}")
print(f"Columns to Export: {df_final.columns.tolist()}")

# Preview to ensure the address and geometry are correct
print("\n--- Data Preview ---")
print(df_final[['center_name', 'address', 'geometry']].head())

# Export to CSV
os.makedirs("output", exist_ok=True)
output_path = "output/jobcenters_berlin_final.csv"
df_final.to_csv(output_path, index=False)

print(f"\n SUCCESS: Final clean data ready at {output_path}")

--- Final Production Audit ---
Total Records: 65
Columns to Export: ['id', 'district_id', 'center_name', 'address', 'postal_code', 'latitude', 'longitude', 'geometry', 'neighborhood', 'district', 'neighborhood_id', 'data_source']

--- Data Preview ---
                       center_name  \
0  Jobcenter Mitte am Leopoldplatz   
1           Arbeitsagentur Spandau   
2        Jobcenter Berlin Neukölln   
3               Agentur für Arbeit   
4               Agentur für Arbeit   

                                             address  \
0  Müllerstraße 147 (Jobcenter Mitte am Leopoldpl...   
1                           Brunsbütteler Damm 75-77   
2                                  Mainzer Straße 27   
3                                   Innungsstraße 40   
4  Agentur für Arbeit, 43-44, Gottlieb-Dunkel-Str...   

                        geometry  
0  POINT (13.3565162 52.5467722)  
1  POINT (13.1865537 52.5337752)  
2  POINT (13.4278868 52.4789752)  
3  POINT (13.3087179 52.5784523)  
4  POIN

In [ ]:
import os

# --- 5.1 SPATIAL DATA PREP ---
# Safety check: Remove any rows with missing shapes to prevent WKT errors
jobcenter_mapped = jobcenter_mapped.dropna(subset=['geometry']).copy()

# Generate the geometry column in WKT format (e.g., POINT (13.4 52.5))
# We name it 'geometry' directly as requested
jobcenter_mapped['geometry'] = jobcenter_mapped['geometry'].apply(
    lambda x: x.wkt if x is not None else None
)

# --- 5.2 FINAL SCHEMA SELECTION ---
final_columns = [
    'id', 
    'district_id', 
    'center_name', 
    'address', 
    'postal_code',
    'latitude', 
    'longitude', 
    'geometry',  
    'neighborhood', 
    'district', 
    'neighborhood_id'
]

# Create the final dataframe and add the source tag
df_final = jobcenter_mapped[final_columns].copy()
df_final['data_source'] = 'OSM_LOR'

# --- 5.3 VERIFICATION & EXPORT ---
print("--- Final Data Audit ---")
print(f"Total Records: {len(df_final)}")
print(f"Columns to Export: {df_final.columns.tolist()}")

# Preview the head to make sure 'geometry' is there
print("\n--- Data Preview ---")
print(df_final[['center_name', 'geometry']].head())

# Export to CSV
os.makedirs("output", exist_ok=True)
output_path = "output/jobcenters_berlin_final.csv"
df_final.to_csv(output_path, index=False)

print(f"\n SUCCESS: Final file with column 'geometry' saved to {output_path}")

--- Final Data Audit ---
Total Records: 65
Columns to Export: ['id', 'district_id', 'center_name', 'address', 'postal_code', 'latitude', 'longitude', 'geometry', 'neighborhood', 'district', 'neighborhood_id', 'data_source']

--- Data Preview ---
                       center_name                       geometry
0  Jobcenter Mitte am Leopoldplatz  POINT (13.3565162 52.5467722)
1           Arbeitsagentur Spandau  POINT (13.1865537 52.5337752)
2        Jobcenter Berlin Neukölln  POINT (13.4278868 52.4789752)
3               Agentur für Arbeit  POINT (13.3087179 52.5784523)
4               Agentur für Arbeit  POINT (13.4114781 52.4565923)

 SUCCESS: Final file with column 'geometry' saved to output/jobcenters_berlin_final.csv


In [ ]:
%pip install sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [ ]:
print(jobcenter_mapped.columns.tolist())

['element', 'id', 'geometry', 'addr:city', 'addr:country', 'addr:housename', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'brand', 'brand:wikidata', 'check_date:opening_hours', 'contact:phone', 'contact:website', 'center_name', 'office', 'opening_hours', 'wheelchair', 'toilets:wheelchair', 'website', 'branch', 'check_date', 'email', 'opening_hours:signed', 'operator', 'phone', 'brand:wikipedia', 'internet_access', 'internet_access:fee', 'internet_access:ssid', 'official_name', 'smoking', 'source', 'contact:email', 'contact:fax', 'description', 'operator:type', 'short_name', 'level', 'note', 'addr:floor', 'building:levels', 'image', 'building', 'building:colour', 'roof:levels', 'roof:shape', 'old_name', 'landuse', 'addr:place', 'heritage', 'heritage:operator', 'heritage:website', 'lda:criteria', 'ref:lda', 'height', 'air_conditioning', 'toilets', 'alt_name', 'name:de', 'type', 'government', 'wikidata', 'latitude', 'longitude', 'address', 'postal_code', 'district_le

In [ ]:
import psycopg2
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings("ignore")

In [ ]:
user_name=''
password=''

In [ ]:
# 1. Configuration (using the stable 127.0.0.1 address)
user_name = 'tigist_hayilemariyam'
password = 'tc3WUbE1DZ6SzYZ'
host = '127.0.0.1' 
port = '5433'
database = 'layereddb'
schema = 'berlin_source_data'
table_name = 'job_centers'

In [ ]:
engine = create_engine(f'postgresql+psycopg2://{user_name}:{password}@{host}:{port}/{database}')

In [ ]:
from sqlalchemy import text

# 1. Define the Corrected Blueprint
# We use 'district_id' for the reference as it's the standard for this database
create_table_query = """
DROP TABLE IF EXISTS berlin_source_data.job_centers CASCADE;

CREATE TABLE berlin_source_data.job_centers (
    id TEXT PRIMARY KEY,
    district_id TEXT NOT NULL,
    center_name TEXT,
    address TEXT,
    postal_code TEXT,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    geometry TEXT,
    neighborhood TEXT,
    district TEXT,
    neighborhood_id TEXT,
    data_source TEXT,
    CONSTRAINT fk_district FOREIGN KEY (district_id) 
        REFERENCES berlin_source_data.districts (district_id) -- Matching the LOR standard
);
"""

# 2. Execute Table Creation
with engine.connect() as conn:
    conn.execute(text(create_table_query))
    conn.commit()
    print(" SUCCESS: The table 'job_centers' has been created!")

# 3. Final Production Upload
try:
    # We use df_final because it contains exactly the columns defined above
    df_final.to_sql(
        name='job_centers',
        con=engine,
        schema='berlin_source_data',
        if_exists='append', 
        index=False
    )
    print(f" MISSION ACCOMPLISHED: {len(df_final)} records are now in AWS production!")
except Exception as e:
    print(f" Upload Error: {e}")
    # Troubleshooting tip: Check if df_final has 'district_id'
    print(f"Your dataframe columns are: {df_final.columns.tolist()}")

✅ SUCCESS: The table 'job_centers' has been created!
🚀 MISSION ACCOMPLISHED: 65 records are now in AWS production!


In [ ]:
# This query asks for every single row
full_check_query = "SELECT * FROM berlin_source_data.job_centers ORDER BY district_id;"

with engine.connect() as conn:
    df_all = pd.read_sql(text(full_check_query), conn)

# This tells the notebook to show all 63 rows without hiding any
with pd.option_context('display.max_rows', None):
    display(df_all)

,id,district_id,center_name,address,postal_code,latitude,longitude,geometry,neighborhood,district,neighborhood_id,data_source
0,3851635472,11001001,Jobcenter,Seydelstraße 2-5,10117,52.510523,13.402383,POINT (13.4023826 52.5105226),Mitte,Mitte,0101,OSM_LOR
1,2782312819,11001001,Job-Point,Alt-Moabit 84,10555,52.525535,13.339522,POINT (13.3395222 52.5255348),Moabit,Mitte,0102,OSM_LOR
2,1346016268,11001001,Zenjob,"Zenjob, Stromstraße, Alt-Moabit, Moabit, Mitte...",10555,52.527579,13.343648,POINT (13.3436479 52.5275788),Moabit,Mitte,0102,OSM_LOR
3,3580091482,11001001,Agentur für Arbeit,Beuthstraße 7,10117,52.510050,13.401957,"POLYGON ((13.4017115 52.5101596, 13.4019428 52...",Mitte,Mitte,0101,OSM_LOR
4,3137117730,11001001,BSM Personalmanagement,"BSM Personalmanagement, Lehrter Straße, Moabit...",10557,52.535615,13.357524,POINT (13.3575242 52.5356149),Moabit,Mitte,0102,OSM_LOR
5,537096787,11001001,Beta gGmbH,Maxstraße 20,13347,52.549239,13.364556,POINT (13.3645563 52.5492387),Wedding,Mitte,0105,OSM_LOR
6,3608024519,11001001,recrew,"recrew, 44, Georgenstraße, Dorotheenstadt, Mit...",10117,52.520063,13.393426,POINT (13.3934256 52.5200629),Mitte,Mitte,0101,OSM_LOR
7,6660665090,11001001,Jobcenter Mitte am Leopoldplatz,Müllerstraße 147 (Jobcenter Mitte am Leopoldpl...,13353,52.546772,13.356516,POINT (13.3565162 52.5467722),Wedding,Mitte,0105,OSM_LOR
8,7211656755,11001001,Agentur Schlag,Joseph-Haydn-Straße 1,10557,52.514923,13.337123,POINT (13.3371231 52.5149233),Hansaviertel,Mitte,0103,OSM_LOR
9,9462705127,11001001,Players Agentur Management,Sophienstraße 21,10178,52.525917,13.400791,POINT (13.4007908 52.525917),Mitte,Mitte,0101,OSM_LOR
